# 06 — Advanced PINN extensions

This notebook extends the baseline with residual-based adaptive collocation and a controlled inverse problem in which the diffusion coefficient $\alpha$ is learned from observations.
These experiments are research demonstrations rather than claims of optimal methodology.

In [ ]:
import torch
from torch import nn
import matplotlib.pyplot as plt
from pinn import MLP,PINNConfig,PINNTrainer,sample_heat_equation,heat_residual,heat_exact_solution
torch.set_default_dtype(torch.float32)
alpha_true=0.1


## 1. Residual-based adaptive refinement

Train a baseline network, evaluate a large candidate pool, and select points with the largest absolute residual. This is a simple RAR-style strategy.

In [ ]:
trainer=PINNTrainer(MLP(hidden_dim=48,hidden_layers=3),PINNConfig(alpha=alpha_true,initial_weight=20,boundary_weight=20,epochs=800,seed=4))
points=sample_heat_equation(1200,300,300,seed=4)
trainer.train(points)
candidate=torch.cat([torch.rand(12000,1)*2-1,torch.rand(12000,1)],1)
r=heat_residual(trainer.model,candidate,alpha_true).detach().abs().squeeze()
chosen=candidate[torch.topk(r,k=400).indices]
print('candidate points:',len(candidate),'selected:',len(chosen))


In [ ]:
plt.figure(figsize=(7,4))
plt.scatter(candidate[:,0],candidate[:,1],s=2,alpha=.06)
plt.scatter(chosen[:,0],chosen[:,1],s=8)
plt.xlabel('x'); plt.ylabel('t'); plt.title('High-residual candidate points'); plt.show()


## 2. Inverse diffusion coefficient

Assume observations of $u(x,t)$ are available and $α$ is unknown. The optimization minimizes both PDE residual and observation misfit while enforcing positive diffusivity with a softplus parameterization.

In [ ]:
class InversePINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model=MLP(hidden_dim=48,hidden_layers=3)
        self.raw_alpha=nn.Parameter(torch.tensor(-2.0))
    @property
    def alpha(self):
        return torch.nn.functional.softplus(self.raw_alpha)
    def forward(self,xt): return self.model(xt)

inverse=InversePINN()
optimizer=torch.optim.Adam(inverse.parameters(),lr=1e-3)


In [ ]:
obs=torch.rand(700,2); obs[:,0]=2*obs[:,0]-1
with torch.no_grad(): y_obs=heat_exact_solution(obs[:,0:1],obs[:,1:2],alpha_true)
interior=sample_heat_equation(1800,300,300,seed=33).interior
trace=[]
for epoch in range(1,1801):
    optimizer.zero_grad(set_to_none=True)
    xf=interior.clone().detach().requires_grad_(True)
    residual=heat_residual(inverse,xf,inverse.alpha)
    physics=residual.square().mean()
    data=(inverse(obs)-y_obs).square().mean()
    loss=physics+20*data
    loss.backward(); optimizer.step()
    trace.append((float(loss),float(inverse.alpha.detach())))
    if epoch==1 or epoch%300==0: print(epoch,float(loss),float(inverse.alpha))


In [ ]:
trace=torch.tensor(trace)
fig,ax=plt.subplots(figsize=(8,4))
ax.plot(trace[:,0]); ax.set_yscale('log'); ax.set_xlabel('epoch'); ax.set_ylabel('loss')
ax2=ax.twinx(); ax2.plot(trace[:,1]); ax2.set_ylabel('estimated alpha')
plt.show()
print('true alpha:',alpha_true,'estimated alpha:',inverse.alpha.item())


## 3. Extension directions

The same computational pattern can estimate spatially varying coefficients, source terms, unknown boundary parameters, or material properties. For real scientific inference, add noise models, uncertainty quantification, identifiability analysis, and independent validation.